# Unigram 语言模型分词器

源码导航：[`core/tokenizer/unigram.py`](../../../core/tokenizer/unigram.py)。

## 1. 理论背景

**Unigram LM** 是 LLaMA、Qwen、DeepSeek 等主流大模型的底层分词算法（通过 SentencePiece 实现）。与 BPE/WordPiece 的"自底向上合并"不同，Unigram 采用**自顶向下裁剪**策略。

### 概率模型

将分词视为一个 Unigram 语言模型：给定候选词表 $\mathcal{V}$，设每个 token $s$ 的概率为 $P(s)$（满足 $\sum_{s \in \mathcal{V}} P(s) = 1$）。对句子 $X$ 的切分 $\mathbf{s} = (s_1, \ldots, s_k)$，对数似然为：

$$\log P(\mathbf{s}) = \sum_{i=1}^{k} \log P(s_i)$$

最优切分 $\mathbf{s}^*$ 定义为使对数似然最大的切分方案，由 **Viterbi 动态规划**求解：

$$dp[i] = \max_{1 \leq l \leq i,\ X[i-l:i] \in \mathcal{V}} \left( dp[i-l] + \log P(X[i-l:i]) \right)$$

### 训练：EM + 裁剪

训练采用 EM 算法与迭代裁剪：
1. **初始化**：从语料中提取所有高频子串构建大型种子词表 $\mathcal{V}_0$（$|\mathcal{V}_0| \gg |\mathcal{V}_\text{target}|$）
2. **E 步**：使用 Viterbi 对语料最优切分，统计每个 token 的使用频次
3. **M 步**：重新估算概率 $P(s) \propto \text{freq}(s)$
4. **裁剪**：删除当前词表中对整体对数似然贡献最低的一批 token（通常 10–20%），确保单字符不被删除
5. 重复至 $|\mathcal{V}| = |\mathcal{V}_\text{target}|$

### SentencePiece 空格归一化

Unigram 通常将文本中的空格替换为可见字符 `▁`（U+2581），并在句首补一个 `▁`，使空格成为普通可见字符参与分词，避免跨词合并问题。

In [ ]:
import math

def viterbi_decode(text, log_probs):
    n = len(text)
    # dp[i] 表示 text[:i] 的最大 log_prob
    dp = [-1e18] * (n + 1)
    # back[i] 记录为了达到最大概率，在位置 i 处选择的 token 长度
    back = [0] * (n + 1)
    
    dp[0] = 0.0
    for i in range(1, n + 1):
        for length in range(1, i + 1):
            sub = text[i-length : i]
            if sub in log_probs:
                lp = log_probs[sub]
                if dp[i-length] + lp > dp[i]:
                    dp[i] = dp[i-length] + lp
                    back[i] = length
                    
    # 反向回溯路径
    tokens = []
    curr = n
    while curr > 0:
        l = back[curr]
        tokens.append(text[curr-l : curr])
        curr -= l
    return tokens[::-1]

# 模拟词表概率 (Log Probs)
# "high" 概率比 "h" + "i" + "g" + "h" 大
log_probs = {
    "h": -5.0, "i": -5.0, "g": -5.0, "high": -2.0,
    "way": -2.0, "highway": -1.0
}

print(f"'highway' 的最优切分: {viterbi_decode('highway', log_probs)}")

## 3. SentencePiece 风格的空格处理

Unigram 通常配合一种特殊的归一化方式：将所有空格替换为 `▁` (U+2581)，并在开头补一个 `▁`。
这样，空格就变成了一个普通可见字符，参与 BPE/Unigram 逻辑。

源码对应：[`_normalize`](../../../core/tokenizer/unigram.py#L48)


In [ ]:
SPACE_MARKER = "\u2581"

def normalize(text):
    return SPACE_MARKER + text.replace(" ", SPACE_MARKER)

print(f"原始: 'hello world'")
print(f"归一化: '{normalize('hello world')}'")

---

## 4. 训练：从“多”到“精”

### 4.1 构造种子词表
训练的第一步是提取语料中所有可能的高频子串，构造一个比最终 Vocab 大得多的初始表。

### 4.2 迭代裁剪
在每一轮迭代中：
1. 运行 Viterbi 统计 Token 出现频次。
2. 计算每个 Token 的贡献。
3. 裁掉贡献最低的一批（例如 20%）。

源码对应：`UnigramTokenizer.train`

---

## 5. 工程实现提示

在源码 [`core/tokenizer/unigram.py`](../../../core/tokenizer/unigram.py) 中：
- **`log_probs`**：存储的是对数概率，相乘变相加，避免浮点数下溢并加速计算。
- **单字符保护**：在裁剪时，永远确保单字符不被删掉，否则会遇到无法编码的文本。
- **Viterbi 复杂度**：对于长度为 $L$ 的句子，复杂度为 $O(L^2)$。

---

## 6. 延伸阅读与参考资料

### 核心论文 (Paper)
- **Unigram LM 提出**: Kudo, 2018. *Subword Regularization*. [arXiv:1804.10959](https://arxiv.org/abs/1804.10959)
- **SentencePiece 实现**: Kudo & Richardson, 2018. [arXiv:1808.06226](https://arxiv.org/abs/1808.06226)

### 优质博客 (Blog)
- **Hugging Face**: *Unigram tokenization*. [NLP Course Chapter 6](https://huggingface.co/learn/nlp-course/chapter6/7)

### 代码库参考 (Code)
- **Google SentencePiece (官方)**: [google/sentencepiece](https://github.com/google/sentencepiece)

---
> 父文档：[← 分词器总览](index.ipynb)